In [ ]:
from dataclasses import dataclass

@dataclass
class ColorContext:
    favourite_color: str = "black"
    least_favourite_color: str = "white"

In [ ]:
from dotenv import load_dotenv

load_dotenv()

### Agent Context -- Immutable

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    context_schema=ColorContext
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What is my favourite color?"),
        ]
    },
    context=ColorContext(),
)

#### Fields in Context can be accessed using Tool Run Time 

In [ ]:
from langchain.tools import tool, ToolRuntime

@tool
def get_favourite_color(runtime: ToolRuntime) -> str:
    """
        Get the favourite color of the user
    """
    return runtime.context.favourite_color

@tool
def get_least_favourite_color(runtime: ToolRuntime) -> str:
    """
        Get the least favourite color of the user
    """
    return runtime.context.least_favourite_color

In [ ]:
agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[get_favourite_color, get_least_favourite_color],
    context_schema=ColorContext
)

In [ ]:
response_with_context = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What is my favourite color?"),
        ]
    },
    context=ColorContext(),
)

In [ ]:
response_with_context

### Agent State -- Mutable and can be updated by agent dynamically while conversing with user

In [ ]:
from langchain.agents import AgentState

class CustomState(AgentState):
    favourite_color: str # unlike contexts, we cannot add default values to fields in state 

In [ ]:
from langchain.tools import ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage


@tool
def update_favourite_color(favourite_color: str, runtime: ToolRuntime) -> Command:
    """Updates the color of the user in the state once they are revealed to agent"""
    return Command(
        update={
            "favourite_color": favourite_color,
            "messages": [
                ToolMessage(
                    content="Successfully updated favourite color",
                    tool_call_id=runtime.tool_call_id,
                )
            ],
        }
    )

In [ ]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

agent = create_agent(
    model="groq:openai/gpt-oss-120b",
    tools=[update_favourite_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

In [ ]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="My favourite color is Red."),
        ]
    },
    {
        "configurable": {
            "thread_id": "7",
        }
    },
)

In [ ]:
response

In [ ]:
response = agent.invoke(
    {
        "messages": [
            HumanMessage(content="What is my favourite color?"),
        ]
    },
    {
        "configurable": {
            "thread_id": "7",
        }
    },
)

In [ ]:
response